# 双留出全基因冻结评估

Jurkat J2 使用 5,000 个基因和 592 个条件。Cross-cell K562 C2 使用 3,352 个基因和 128 个条件。每个条件使用 300 个固定输入对照。缺失输出列由该条件对照的同一基因均值填补，Systema 的参考只用训练扰动。每份基因清单按数据集的原始表达列顺序记录。

In [ ]:
from pathlib import Path
import hashlib
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

root = Path.cwd()
plan = json.loads((root / "comparison-inputs.json").read_text())
scenario = plan["scenario"]
gene_count, condition_count = {"J2": (5000, 592), "C2": (3352, 128)}[scenario]
identifiers = (
    "condition_ids_sha256", "paired_rows_sha256", "test_panel_gene_order_sha256",
    "systema_reference_condition_ids_sha256",
)
reference_field = "systema_reference_sha256" if scenario == "J2" else "systema_reference_train_rows_sha256"
coverage_records = {}
keys = [
    "txpert_macro_pearson_delta_all", "trishift_pearson_delta_all", "systema_pearson_all",
    "mean_mse", "retrieval",
]
labels = ["TxPert Δ Pearson", "TriShift Δ Pearson", "Systema Pearson", "Mean MSE", "Retrieval"]
tables, identities = {}, set()
order = None
for name, filename in plan["results"].items():
    path = (root / filename).resolve()
    result = json.loads(path.read_text())
    assert result["scenario"] == scenario and result["status"] == "evaluated" and result["systema_reference_scope"] == "train_only"
    assert (result["conditions"], result["test_controls_per_condition"],
            result["test_panel_genes"], result["test_updates"]) == (condition_count, 300, gene_count, 0)
    assert result["score_scope"] == "full_gene_axis" and result["formal_eligible"]
    identities.add(tuple(result[k] for k in (*identifiers, reference_field)))
    gene_receipt = (path.parent / "evaluation_genes.json").read_bytes()
    assert hashlib.sha256(gene_receipt).hexdigest() == result["evaluation_genes_sha256"]
    gene_panel = json.loads(gene_receipt)
    genes = gene_panel["genes_in_expression_order"]
    native = set(gene_panel["native_genes_in_expression_order"])
    imputed = set(gene_panel["imputed_genes_in_expression_order"])
    assert gene_panel["imputation_rule"] == "mean_of_fixed_condition_input_controls"
    assert len(genes) == len(set(genes)) == gene_count and native.isdisjoint(imputed)
    assert native | imputed == set(genes)
    assert len(native) == result["native_prediction_genes"]
    assert len(imputed) == result["imputed_prediction_genes"]
    coverage_records[name] = {"native": len(native), "imputed": len(imputed)}
    raw = (path.parent / "conditions.jsonl").read_bytes()
    assert hashlib.sha256(raw).hexdigest() == result["conditions_file_sha256"]
    records = [json.loads(line) for line in raw.splitlines()]
    ids = [record["condition_id"] for record in records]
    assert len(set(ids)) == len(ids) == condition_count
    if order is None:
        order = ids
    assert order == ids
    frame = pd.DataFrame(
        [{**{m["metric_id"]: m["value"] for m in row["results"]},
          "mean_mse": (row.get("response_metrics") or {}).get("condition_mean_mse", row.get("mean_mse")),
          "retrieval": (row.get("response_metrics") or {}).get("txpert_normalized_retrieval", row.get("retrieval"))} for row in records],
        index=pd.Index(ids, name="condition"),
    )[keys]
    summary = {m["metric_id"]: m["macro_mean"] for m in result["pearson"]}
    expected = [*map(summary.get, keys[:3]), result["mean_mse"], result["retrieval_mean"]]
    np.testing.assert_allclose(frame.mean().to_numpy(), expected, rtol=1e-8, atol=1e-10)
    tables[name] = frame
coverage = pd.DataFrame.from_dict(coverage_records, orient="index")
coverage.to_csv(root / "gene-coverage.csv", index_label="model")
display(coverage)
assert len(identities) == 1 and len(tables) >= 2
assert plan["reference"] in tables
names = list(tables)
all_rows = pd.concat(tables, names=["model"]).reset_index()
all_rows.to_csv(root / "paired-condition-metrics.csv", index=False)
summary = all_rows.groupby("model", sort=False)[keys].mean()
summary.to_csv(root / "metrics.csv")
display(summary.rename(columns=dict(zip(keys, labels, strict=True))).round(5))

fig, axes = plt.subplots(2, 3, figsize=(14, 7), layout="constrained")
colors = plt.colormaps["tab10"](np.arange(len(names)))
for i, (ax, key, label) in enumerate(zip(axes.flat, keys, labels)):
    ax.boxplot([tables[name][key].dropna() for name in names], tick_labels=names, showfliers=False)
    ax.scatter(np.arange(1, len(names) + 1), summary[key], color=colors, marker="D", s=22)
    ax.set(title=label + (" ↓" if i == 3 else " ↑"))
    ax.tick_params(axis="x", rotation=28)
coverage.plot.bar(stacked=True, ax=axes.flat[-1], color=["#2B6F9D", "#DE7930"], rot=28)
axes.flat[-1].set(title="Native and imputed genes", ylim=(0, gene_count * 1.1), xlabel="")
fig.savefig(root / "full-gene-metric-distributions.png", dpi=170)
plt.show()

rng = np.random.default_rng(20261125)
indices = rng.integers(0, condition_count, size=(10000, condition_count))
reference = plan["reference"]
comparisons = []
for name in names:
    if name == reference:
        continue
    for key in keys:
        diff = (tables[name][key] - tables[reference][key]).to_numpy()
        valid = np.isfinite(diff)
        if not valid.all():
            raise ValueError("paired confidence intervals require finite pairs for every condition")
        low, high = np.quantile(diff[indices].mean(1), [.025, .975])
        comparisons.append(dict(model=name, metric=key, paired_conditions=len(diff),
                                delta=diff.mean(), ci_low=low, ci_high=high))
paired = pd.DataFrame(comparisons)
paired.to_csv(root / "paired-bootstrap.csv", index=False)
fig, axes = plt.subplots(2, 3, figsize=(14, 7), layout="constrained")
for ax, key, label in zip(axes.flat, keys, labels):
    data = paired[paired.metric == key]
    y = np.arange(len(data))
    ax.errorbar(data.delta, y, xerr=[data.delta - data.ci_low, data.ci_high - data.delta],
                fmt="o", color="#285d9f")
    ax.set(yticks=y, yticklabels=data.model, title=label)
    ax.axvline(0, lw=1, color=".5")
axes.flat[-1].axis("off")
fig.suptitle(f"Paired difference versus {reference} · 95% conditional bootstrap interval")
fig.savefig(root / "paired-condition-bootstrap.png", dpi=170)
plt.show()

# Identify unfavorable conditions by a within-model rank on each endpoint.
# Metrics keep their original signed values in the exported table.
rows = []
for name, frame in tables.items():
    ranks = frame.rank(pct=True)
    ranks["mean_mse"] = frame["mean_mse"].rank(pct=True, ascending=False)
    rows.append(ranks.mean(axis=1).rename(name))
diagnosis = pd.concat(rows, axis=1).assign(mean_percentile=lambda x: x.mean(axis=1))
selected = diagnosis.sort_values("mean_percentile").head(20)
selected.to_csv(root / "hardest-conditions.csv")
fig, ax = plt.subplots(figsize=(8, 7), layout="constrained")
heat = ax.imshow(selected[names], vmin=0, vmax=1, aspect="auto", cmap="viridis")
ax.set(xticks=range(len(names)), xticklabels=names, yticks=range(len(selected)),
       yticklabels=[label.removesuffix("+ctrl") for label in selected.index],
       title="20 lowest-ranked conditions across models")
fig.colorbar(heat, ax=ax, label="Mean within-model metric percentile")
fig.savefig(root / "hardest-conditions.png", dpi=170)
plt.show()